# Lab 03: AQE and how operators run

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>1.4 Adaptive Query Execution</b></span><br><span style="color:#d93025"><b>1.9 count() vs approx_count_distinct()</b></span><br><span style="color:#d93025"><b>1.14 How count(distinct) works</b></span><br><span style="color:#d93025"><b>1.15 How avg() works</b></span><br><span style="color:#d93025"><b>1.16 How sorting works</b></span><br><span style="color:#d93025"><b>3.11 spark.sql.shuffle.partitions</b></span>

**Goal:** see how aggregates and sorts run across partitions, and what Spark changes at runtime when AQE is on. The last section is window-function practice.

## Data information
A small online shop with **one table**. The setup cell below creates it in `data/orders.parquet` (1,000,000 orders from January to March 2024; seeded, so everyone gets the same rows):
* *order_id*; *customer_id* (empty for guest checkout), *customer_name*, *country*, *segment* (`retail`, `business` or `guest`)
* *product_id*, *category* (8 categories), *price* (price of one item)
* *quantity* (1 to 5) and *amount* = quantity × price, in dollars
* *status*: `paid`, `shipped`, `cancelled` or `returned`; *payment_method*: `card`, `cash` or `wallet`
* *order_ts* and *order_date*: when the order was placed; *tracking_code*: a 32-character code

Customer 1 is "Big Reseller Ltd": a business customer who orders in bulk (quantity 5) and places 30% of all orders. Some labs make a small lookup table from this one table (for example one row per customer); the notebook says when.

## How to work
* Write your answer in the code cell under each question and run it. The answers are at the very end of this notebook (collapsed): open one only after you tried. Each answer shows the code and the result of the reference run, so you can compare your numbers.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then open the answer of that question.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [Japila: HashAggregateExec](https://books.japila.pl/spark-sql-internals/physical-operators/HashAggregateExec/)
* [DataFrame.explain](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)
* [functions.approx_count_distinct](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.approx_count_distinct.html)
* [HyperLogLog (Wikipedia)](https://en.wikipedia.org/wiki/HyperLogLog)
* [Japila: SortExec](https://books.japila.pl/spark-sql-internals/physical-operators/SortExec/)
* [ORDER BY / SORT BY](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-orderby.html)
* [Adaptive Query Execution](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution)
* [AQE (blog)](https://www.databricks.com/blog/2020/05/29/adaptive-query-execution-speeding-up-spark-sql-at-runtime.html)
* [Window](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.html)
* [functions.row_number](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.row_number.html)
* [Window functions (SQL)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)
* [functions.lag](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lag.html)
* [Window.rowsBetween](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.rowsBetween.html)
* [functions.first](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.first.html)
* [functions.max_by](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.max_by.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop with ONE table, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    cust = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                        # guest checkout: no customer
             .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                           # the big reseller: 30% of orders
             .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    name = F.when(cust == 1, "Big Reseller Ltd").when(cust.isNotNull(), F.concat(F.lit("Customer "), cust.cast("string")))
    country = _pick(F.when(cust.isNotNull(), _u(cust, 1)).otherwise(_u(i, 31)), COUNTRIES)
    segment = F.when(cust.isNull(), "guest").when((cust == 1) | (_u(cust, 3) < 0.2), "business").otherwise("retail")
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    category = F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (product % 8 + 1).cast("int"))
    price = F.round((product * 37) % 195 + 5.99, 2)
    quantity = F.when(cust == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), cust.alias("customer_id"), name.alias("customer_name"), country.alias("country"), segment.alias("segment"),
        product.alias("product_id"), category.alias("category"), price.alias("price"), quantity.alias("quantity"),
        F.round(quantity * price, 2).alias("amount"), status.alias("status"), payment.alias("payment_method"),
        ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "customer_name", "country", "segment", "product_id", "category", "price", "quantity",
                     "amount", "status", "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day03_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session()
def _data_ok():
    """True if the single-table shop data is already there (an older three-table version is rebuilt)."""
    try:
        return "customer_name" in spark.read.parquet("data/orders.parquet").columns and os.path.exists("data/monthly/orders_2024-03.parquet")
    except Exception:
        return False

if not _data_ok():
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)


Load the data the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")

## A. How aggregates and sorts run (AQE is off)

**3.1 How many shuffle `Exchange` nodes does each query need, and which special operators show up?** Use `count_nodes(df, "Exchange")`.
* *exchanges* = dict with the keys `avg` (`groupBy("status")` + `avg("amount")`), `count_distinct` (`countDistinct("customer_id")` per status), `approx_distinct` (`approx_count_distinct("customer_id")` per status) and `order_by` (`orderBy("amount")`)
* *has_expand*: does a query with two different `countDistinct` columns per status contain an `Expand` node? (True/False)
* *uses_object_agg*: does `collect_list("amount")` per status use `ObjectHashAggregate`? (True/False)

Put `[exchanges, has_expand, uses_object_agg]` in *result*. Guess first.

**3.2 How close is `approx_count_distinct` to the exact `countDistinct` of *customer_id*, per *status*?**
* *exact_counts* = dict `{status: exact count}`
* the relative error of `approx_count_distinct("customer_id", 0.05)` must be under 10% and of `approx_count_distinct("customer_id", 0.01)` under 3% for every status (*errors_ok*, True/False)

Keep the exact and the approximate counts in two separate aggregations and join them: in one `agg`, Spark builds an approximate sketch for every (status, customer) pair and can run out of memory.

**3.3 Why is the average of averages wrong?** Compute the overall average *amount* of all orders (*avg_all*) and the average of the per-customer averages, ignoring guests (*avg_of_avgs*), both rounded to 2 decimals. Customer 1 orders in bulk and places 30% of all orders.

**3.4 How does a global sort run, and what does `limit` change?**
* *jobs_full*: jobs started by sorting all orders by *amount* and writing with `noop`
* *top10* = the 10 largest orders (*amount* descending, ties by *order_id*); *jobs_limit*: jobs started by collecting it
* *uses_take_ordered*: does the plan of *top10* contain `TakeOrderedAndProject`? (True/False)

Put `[jobs_full, jobs_limit, uses_take_ordered]` in *result*. Guess first: why does a global sort need an extra sampling job?

## B. AQE and the shuffle partitions setting

**3.5 What does AQE change for `q = orders.groupBy("customer_id").agg(F.sum("amount"))`?** Use `last_stage_tasks` (the number of tasks of the last stage):
* *tasks_off*: AQE off, default shuffle partitions (200)
* *tasks_2000*: AQE off, `spark.sql.shuffle.partitions` set to 2000
* *tasks_aqe*: AQE on (`spark.sql.adaptive.enabled`), still 2000 partitions; build *q2* again and collect it
* *coalesced*: does the final plan of *q2* contain `coalesced`? (True/False)

Then set AQE off and the partitions back to 200.

## C. Coding practice: window functions

Three problems with `Window`: rank within a group, running totals and neighbours, first and last row. Write them from scratch. Break ties explicitly so your answer is always the same.

**3.6 Which are the 3 best-selling products (by revenue of paid orders) in every product category?**
* revenue per product = sum of *amount* of paid orders, per *category* and *product_id*
* rank the products inside each category by revenue (largest first, ties by *product_id*) and keep ranks 1 to 3
* *top_products* = list of `(category, product_id, revenue)` with revenue rounded to 2 decimals, sorted by *category*, then rank

**3.7 Daily revenue of paid orders: how does January develop?**
* *daily* = revenue per *order_date* (paid orders only)
* add *running* (running total over the days, in date order) and *change* (revenue minus the revenue of the previous day)
* *jan_total*: the running total on `2024-01-31` (2 decimals)
* *best_day*: the date (as text `YYYY-MM-DD`) with the largest *change* in all three months

Put `[jan_total, best_day]` in *result*.

**3.8 Which customers spend more per order over time?** For every customer (no guests) with at least 2 orders, compare the *amount* of their first order (by *order_ts*, ties by *order_id*) with the amount of their last order.
* *growth* = last amount minus first amount
* *top_growth* = the 3 customers with the largest *growth* (ties by *customer_id*) as `[(customer_id, growth), ...]` with growth rounded to 2 decimals

---
Done? In the Theory part of Day 3, answer the red checklist topics aloud, with the plans and numbers from this lab as evidence.

## Answers
Try every question yourself first. When you are stuck, click the arrow of that question below to open its answer, then compare it with your own. Each answer is the reference solution with the result of its run; copy the code into the cell above to run it.

<details>
<summary><b>3.1</b> &nbsp;How many shuffle Exchange nodes does each query need, and which special operators show up?</summary>

```python
def exchanges_of(df):
    return count_nodes(df, "Exchange")["Exchange"]

exchanges = {
    "avg": exchanges_of(orders.groupBy("status").agg(F.avg("amount"))),
    "count_distinct": exchanges_of(orders.groupBy("status").agg(F.countDistinct("customer_id"))),
    "approx_distinct": exchanges_of(orders.groupBy("status").agg(F.approx_count_distinct("customer_id"))),
    "order_by": exchanges_of(orders.orderBy("amount")),
}
two = orders.groupBy("status").agg(F.countDistinct("customer_id"), F.countDistinct("product_id"))
has_expand = count_nodes(two, "Expand")["Expand"] > 0
uses_object_agg = count_nodes(orders.groupBy("status").agg(F.collect_list("amount")), "ObjectHashAggregate")["ObjectHashAggregate"] > 0
result = [exchanges, has_expand, uses_object_agg]
result
```

Result:

```
[{'avg': 1, 'count_distinct': 2, 'approx_distinct': 1, 'order_by': 1},
 True,
 True]
```

</details>

<details>
<summary><b>3.2</b> &nbsp;How close is approx_count_distinct to the exact countDistinct of customer_id, per status?</summary>

```python
exact = orders.groupBy("status").agg(F.countDistinct("customer_id").alias("exact"))
approx = orders.groupBy("status").agg(F.approx_count_distinct("customer_id", 0.05).alias("a05"), F.approx_count_distinct("customer_id", 0.01).alias("a01"))
rows = exact.join(approx, "status").collect()
exact_counts = {r["status"]: r["exact"] for r in rows}
errors_ok = all(abs(r["a05"] - r["exact"]) / r["exact"] < 0.10 and abs(r["a01"] - r["exact"]) / r["exact"] < 0.03 for r in rows)
exact_counts, errors_ok
```

Result:

```
({'shipped': 10000, 'cancelled': 9995, 'paid': 10000, 'returned': 9952}, True)
```

</details>

<details>
<summary><b>3.3</b> &nbsp;Why is the average of averages wrong?</summary>

```python
avg_all = round(orders.agg(F.avg("amount")).first()[0], 2)
per_customer = orders.filter(F.col("customer_id").isNotNull()).groupBy("customer_id").agg(F.avg("amount").alias("a"))
avg_of_avgs = round(per_customer.agg(F.avg("a")).first()[0], 2)
avg_all, avg_of_avgs
```

Result:

```
(370.58, 309.9)
```

</details>

<details>
<summary><b>3.4</b> &nbsp;How does a global sort run, and what does limit change?</summary>

```python
jobs_full = jobs_run(lambda: noop(orders.orderBy("amount")))
top10 = orders.orderBy(F.desc("amount"), "order_id").limit(10)
jobs_limit = jobs_run(top10.collect)
uses_take_ordered = count_nodes(top10, "TakeOrderedAndProject")["TakeOrderedAndProject"] == 1
result = [jobs_full, jobs_limit, uses_take_ordered]
result
```

Result:

```
[2, 1, True]
```

</details>

<details>
<summary><b>3.5</b> &nbsp;What does AQE change for q = orders.groupBy("customer_id").agg(F.sum("amount"))?</summary>

```python
q = orders.groupBy("customer_id").agg(F.sum("amount"))
tasks_off = last_stage_tasks(lambda: noop(q))
spark.conf.set("spark.sql.shuffle.partitions", 2000)
tasks_2000 = last_stage_tasks(lambda: noop(q))
spark.conf.set("spark.sql.adaptive.enabled", "true")
q2 = orders.groupBy("customer_id").agg(F.sum("amount"))
tasks_aqe = last_stage_tasks(q2.collect)
coalesced = "coalesced" in plan(q2, "simple")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 200)
tasks_off, tasks_2000, tasks_aqe, coalesced
```

Result:

```
(200, 2000, 1, True)
```

</details>

<details>
<summary><b>3.6</b> &nbsp;Which are the 3 best-selling products (by revenue of paid orders) in every product category?</summary>

```python
revenue = (orders.filter(F.col("status") == "paid")
           .groupBy("category", "product_id").agg(F.sum("amount").alias("revenue")))
w = Window.partitionBy("category").orderBy(F.desc("revenue"), "product_id")
ranked = revenue.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= 3)
top_products = [(r["category"], r["product_id"], round(r["revenue"], 2)) for r in ranked.orderBy("category", "rank").collect()]
top_products
```

Result:

```
[('beauty', 21, 1914167.32),
 ('beauty', 5, 1884307.34),
 ('beauty', 189, 1693096.69),
 ('books', 184, 1847075.61),
 ('books', 200, 1829875.19),
 ('books', 168, 1765475.25),
 ('garden', 42, 1986558.12),
 ('garden', 26, 1861101.0),
 ('garden', 10, 1714518.27),
 ('kitchen', 100, 1921877.94),
 ('kitchen', 84, 1871378.98),
 ('kitchen', 68, 1822811.84),
 ('music', 142, 1862661.96),
 ('music', 158, 1843629.41),
...
```

</details>

<details>
<summary><b>3.7</b> &nbsp;Daily revenue of paid orders: how does January develop?</summary>

```python
daily = (orders.filter(F.col("status") == "paid").groupBy("order_date").agg(F.sum("amount").alias("revenue")))
w = Window.orderBy("order_date")
trend = (daily.withColumn("running", F.sum("revenue").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
              .withColumn("change", F.col("revenue") - F.lag("revenue").over(w)))
jan_total = round(trend.filter(F.col("order_date") == "2024-01-31").first()["running"], 2)
best_day = str(trend.orderBy(F.desc("change")).first()["order_date"])
result = [jan_total, best_day]
result
```

Result:

```
[69159498.33, '2024-01-24']
```

</details>

<details>
<summary><b>3.8</b> &nbsp;Which customers spend more per order over time?</summary>

```python
w = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")
wf = w.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
first_last = (orders.filter(F.col("customer_id").isNotNull())
              .withColumn("first_amount", F.first("amount").over(wf))
              .withColumn("last_amount", F.last("amount").over(wf))
              .withColumn("n", F.count("*").over(wf))
              .filter(F.col("n") >= 2)
              .select("customer_id", (F.col("last_amount") - F.col("first_amount")).alias("growth")).distinct())
top_growth = [(r["customer_id"], round(r["growth"], 2)) for r in first_last.orderBy(F.desc("growth"), "customer_id").limit(3).collect()]
top_growth
```

Result:

```
[(2913, 977.97), (8032, 961.97), (4364, 959.96)]
```

</details>